In [ ]:
###--- Load libraries and set the location path for analysis data ---###

In [ ]:
# Environment setup
import numpy as np
import scanpy as sc
import pandas as pd
import scipy.io
import matplotlib as mpl
import batchglm.api as glm
import diffxpy.api as de
import decoupler as dc

from matplotlib import rcParams
import bbknn
import os
import sys
import scipy
import seaborn as sns
import scipy.io as sio
import scanpy.external as sce
import matplotlib.pyplot as plt
import scipy.sparse as sp
import scrublet as scr
import scipy.stats as stats


In [ ]:
sc.settings.verbosity = 2  # show logging output
sc.settings.dir = "scRNA_Glyco/pseudo"
sc.settings.autosave = True  # save figures, do not show them
sc.settings.figdir = "scRNA_Glyco/pseudo/figure"
sc.settings.set_figure_params(dpi=200, format="pdf", dpi_save=400)

In [ ]:
# Data Loading
adata_celltypist = sc.read("clustering.h5ad")
# Start with Raw data
adata_celltypist.X = adata_celltypist.layers["counts"].copy()

# List of samples
comparison_id= ['CAR1','CAR2','CAR3','Tr2DG1','Tr2DG2','Tr2DG3','TrTUN1','TrTUN2','TrTUN3']

# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask = adata_celltypist.obs['sample'].isin(comparison_id)
adata_celltypist_my = adata_celltypist[boolean_mask, :]


In [ ]:
###--- Generation of pseudo-bulk profiles ---###

In [ ]:
# Get filtered pseudo-bulk profile
pdata = dc.get_pseudobulk(adata_celltypist, sample_col='sample',
    groups_col='leiden', layer='counts',
    mode='sum', min_cells=10, min_counts=1000
)
pdata

In [ ]:
pdata.obs['sample']

In [ ]:
pdata.obs['leiden']

In [ ]:
# Pseudo-bulk profile gene filtering

In [ ]:
# Import DESeq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

## Subsetting the cell type of interest  ##
cluster_idex = '10'
sp_cells = pdata[pdata.obs['leiden'] == cluster_idex].copy()

# Obtain genes that pass the thresholds
genes = dc.filter_by_expr(sp_cells, group='label', min_count=5, min_total_count=15)

# Filter by these genes
sp_cells = sp_cells[:, genes].copy()
sp_cells

## Contrast between conditions ##

# Build DESeq2 object
dds = DeseqDataSet(
    adata=sp_cells,
    design_factors=['donor','label'],
    ref_level=['label', 'CAR'],
    refit_cooks=True,
)

# Compute LFCs
dds.deseq2()

# Extract contrast
comparison = 'Tr2DG'
stat_res = DeseqStats(dds, contrast=["label", comparison, 'CAR'], n_cpus=8, cooks_filter= False, independent_filter=False)
stat_res.summary()

# Shrink LFCs
stat_res.lfc_shrink()

# Extract results
results_df = stat_res.results_df
results_df
results_df.to_csv(f"pdeuso/de_deseq2_{cluster_idex}_cluster_vs{comparison}.csv")

dc.plot_volcano_df(results_df, x='log2FoldChange', y='padj', top=20, save= f"pdeuso/de_deseq2_{cluster_idex}_cluster_vs{comparison}.pdf")

In [ ]:
###--- Functional enrichment of biological terms ---###

In [ ]:
# Retrieve The Molecular Signatures Database (MSigDB) resource
msigdb = dc.get_resource('MSigDB')
# Extract unique categories from the 'collection' column
unique_collections = list(msigdb['collection'].cat.categories)
print(unique_collections)

In [ ]:
# List of desired categories
path_categories = ['hallmark', 'kegg_pathways', 'reactome_pathways']

# Filter msigdb DataFrame based on categories in path_categories list
msigdb_r = msigdb[msigdb['collection'].isin(path_categories)]

# Remove duplicated entries
msigdb_r = msigdb_r[~msigdb_r.duplicated(['geneset', 'genesymbol'])]

msigdb_r

In [ ]:
#import data 
cluster_idex = '14'
comparison = 'Tr2DG'

file_path = f"pdeuso/de_deseq2_{cluster_idex}_cluster_vs{comparison}.csv"
results_df = pd.read_csv(file_path, index_col=0)


# Infer enrichment with ora using significant degs
top_genes_down = results_df[(results_df['log2FoldChange'] < -0.5) & (results_df['padj'] < 0.05)]
top_genes_up = results_df[(results_df['log2FoldChange'] > 0.5) & (results_df['padj'] < 0.05)]

# Run ora top_genes_up 
enr_pvals_up = dc.get_ora_df(
    df=top_genes_up,
    net=msigdb_r,
    source='geneset',
    target='genesymbol',
)
enr_pvals_up.sort_values('FDR p-value', ascending=True).head(15)
enr_pvals_up.to_csv(f"pdeuso/up_Enrich_{cluster_idex}_cluster_vs{comparison}.csv")


# Sorting and assigning new column for upregulated genes
enr_pvals_up_fdr = enr_pvals_up[enr_pvals_up['FDR p-value'] < 0.1].copy()
enr_pvals_up_fdr["-log10(FDR)"] = -np.log10(enr_pvals_up_fdr["FDR p-value"])
enr_pvals_up_fdr["log10(score)"] = np.log10(enr_pvals_up_fdr["Combined score"])

# Plot
upplot = dc.plot_dotplot(
        enr_pvals_up_fdr.sort_values('Combined score', ascending=False).head(10),  
        x='-log10(FDR)',
        y='Term',
        s='log10(score)',
        c='-log10(FDR)',
        scale=1,    
        cmap='copper',
        title=f"up_Enrich_{cluster_idex}_cluster_vs{comparison}",
        save=f"pdeuso/up_Enrich_{cluster_idex}_cluster_vs{comparison}.pdf"
    )


# Run ora top_genes_down
enr_pvals_down = dc.get_ora_df(
    df=top_genes_down,
    net=msigdb_r,
    source='geneset',
    target='genesymbol',
)
enr_pvals_down.to_csv(f"pdeuso/down_Enrich{cluster_idex}_cluster_vs{comparison}.csv")
enr_pvals_down.sort_values('FDR p-value', ascending=True).head(15)

# Sorting and assigning new column for downregulated genes
enr_pvals_down_fdr = enr_pvals_down[enr_pvals_down['FDR p-value'] < 0.1].copy()
enr_pvals_down_fdr["-log10(FDR)"] = -np.log10(enr_pvals_down_fdr["FDR p-value"])
enr_pvals_down_fdr["log10(score)"] = np.log10(enr_pvals_down_fdr["Combined score"])

# Plot
downplot = dc.plot_dotplot(
        enr_pvals_down_fdr.sort_values('Combined score', ascending=False).head(10),  
        x='-log10(FDR)',
        y='Term',
        s='log10(score)',
        c='-log10(FDR)',
        scale=1,
        cmap='copper',
        title=f"down_Enrich_{cluster_idex}_cluster_vs{comparison}",
        save=f"pdeuso/down_Enrich_{cluster_idex}_cluster_vs{comparison}.pdf"
    )

In [ ]:
###--- Gene signatures ---###

In [ ]:
#  UPR signature
gene_list_UPR = []
gene_list_UPR.extend(['ABCA7', 'AGR2', 'ATF4', 'BOK', 'DDIT3', 'EIF2AK3', 'EIF2S1', 'HSPA5', 'NCK1', 'NCK2', 'NFE2L2',
                      'PPP1R15A', 'PPP1R15B', 'PTPN1', 'PTPN2', 'QRICH1', 'TMED2', 'TMEM33', 'BAK1', 'BAX', 'BCL2L11',
                      'BFAR', 'COPS5', 'DAB2IP', 'DDRGK1', 'DNAJB9', 'DNAJC10', 'ERN1', 'ERN2', 'FICD', 'PARP16', 'UFL1',
                      'VAPB', 'XBP1'])

In [ ]:
import scipy.stats as stats
# List of annotation categories
comparison_id= ['CAR1','CAR2','CAR3','Tr2DG1','Tr2DG2','Tr2DG3','TrTUN1','TrTUN2','TrTUN3']

# Create a boolean series indicating whether each annotation is in the list of desired categories
boolean_mask = adata.obs['sample'].isin(comparison_id)
subset_adata_my_cmp = adata[boolean_mask, :]

sc.tl.score_genes(adata, gene_list_UPR, ctrl_size=50, gene_pool=None, n_bins=25, score_name='score_UPR', random_state=0, copy=False, use_raw=None)
groups = subset_adata_my_cmp.obs['label'].unique()

p_values = {}

for group in groups:
    group_data = subset_adata_my_cmp.obs['score_UPR'][subset_adata_my_cmp.obs['label'] == group]
    other_groups = [g for g in groups if g != group]
    for other_group in other_groups:
        other_group_data = subset_adata_my_cmp.obs['score_UPR'][subset_adata_my_cmp.obs['label'] == other_group]
        _, p_value = stats.mannwhitneyu(group_data, other_group_data)
        p_values[f'{group}_vs_{other_group}'] = p_value

# Create a violin plot
plt.figure(figsize=(10, 6))
sns.violinplot(x='label', y='score_UPR', data=subset_adata_my_cmp.obs)
plt.savefig('_UPR_distribution.pdf', bbox_inches='tight')

In [ ]:
###--- Functional enrichment of biological terms ---###

In [ ]:
# Selected cluster
celltype_condition = "4"
# extract scores
t_stats = (
    # Get dataframe of DE results for condition vs. rest
    sc.get.rank_genes_groups_df(adatas, celltype_condition, key="dea_leiden")
    # Subset to highly variable genes
    .set_index("names")
    .loc[adatas.var["highly_variable"]]
    # Sort by absolute score
    .sort_values("scores", key=np.abs, ascending=False)
    # Format for decoupler
    [["scores"]]
    .rename_axis(["path"], axis=1)
)
t_stats

In [ ]:
# Running GSEA
scores, norm, pvals = dc.run_gsea(
    t_stats.T,
    msigdb_r,
    source="geneset",
    target="genesymbol",
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Build GSEA results
gsea_results = (
    pd.concat({"score": scores.T, "norm": norm.T, "pval": pvals.T}, axis=1)
      .droplevel(1, axis=1)
)

# Filter, add metrics, and order once by p-value
filtered_results = (
    gsea_results
      .query("norm > 0 and pval < 0.2")
      .assign(
          **{"-log10(pval)": lambda d: -np.log10(d["pval"].clip(lower=1e-300))},
          geneset=lambda d: d.index
      )
      .sort_values("pval")
)

# Dotplot
dc.plot_dotplot(
    df=filtered_results,
    x="norm",
    y="geneset",
    s="-log10(pval)",
    c="-log10(pval)",
    scale=0.5,
    cmap="viridis"
)

plt.savefig("enrichment_hallmark.pdf", format="pdf", bbox_inches="tight")
plt.close()
